# 06 — полный development для фиксированной LoRA 0,25

**Google Colab → Среда выполнения → Сменить среду выполнения → T4 GPU.**
Используйте новую сессию. Выполняйте восемь кодовых ячеек сверху вниз.

Загрузите исходный архив **sft_v3_pilot_qwen3_4b_20261118T113449101907.zip**
с весами. Архив с оценками судей не подходит. Все остальные данные
автоматически загружаются из закреплённого коммита GitHub и проверяются по SHA-256.

План: 32 прежних baseline + 8 прежних ответов LoRA 0,25 + **24 новых ответа**.
Дополнительно выполняются шесть контрольных генераций. Обучения и запросов к судье нет.
Условия прежние: 768 токенов, sampling, индивидуальный seed каждого вопроса.
Это расширенный **development**, не независимый финальный тест.

В конце скачиваются **два ZIP**: полный результат и отдельный пакет из 24 заданий судье.
Разрешите браузеру скачивание нескольких файлов. Сохраните также выполненный IPYNB.


## 1. Установка версий прежнего успешного T4-прогона

Если появится требование перезапустить Python: **Среда выполнения → Перезапустить сеанс**,
затем снова выполняйте ячейки с первой. Менять версии для обхода проверки не нужно.


In [ ]:
import importlib.metadata
import os
import subprocess
import sys
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
PINS = {'transformers': '5.22.1', 'tokenizers': '0.23.3', 'huggingface-hub': '2.4.0', 'accelerate': '1.12.0', 'bitsandbytes': '0.50.2', 'peft': '0.21.2', 'safetensors': '0.8.0', 'numpy': '2.4.6'}
TORCH_PIN = '2.14.1+cu130'
def installed(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
changed = {p for p,v in PINS.items() if installed(p) != v}
if installed('torch') != TORCH_PIN:
    changed.add('torch')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'torch=='+TORCH_PIN, '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
if changed & set(PINS):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    *[p+'=='+v for p,v in PINS.items()]], check=True)
expected_versions = dict(PINS, torch=TORCH_PIN)
loaded = [p for p,v in expected_versions.items()
          if p.replace('-', '_') in sys.modules
          and (p in changed or getattr(sys.modules[p.replace('-', '_')], '__version__', v) != v)]
if loaded:
    raise RuntimeError('Перезапустите сеанс Python после установки: '+', '.join(loaded))
assert installed('torch') == TORCH_PIN
assert all(installed(p)==v for p,v in PINS.items())
print('Версии установлены. Генерация ещё не запускалась.')


## 2. Источники, параметры и папка результата

Можно оставить настройки как есть. Каждый ответ сразу сохраняется в `/content`.
Для защиты от потери сессии поставьте `PERSIST_TO_DRIVE=True`: появится запрос
подключить ваш Google Drive, результаты будут записываться туда после каждого ответа.

После обрыва сессии результаты из `/content` исчезают. Если использовали Drive,
укажите полный путь прежней папки запуска в `RESUME_DIR` и выполните ноутбук заново.
Во время живой сессии для продолжения повторяют только ячейку 7.


In [ ]:
import contextlib, hashlib, inspect, json, math, platform, random, shutil, sys, time, traceback, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
import peft
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig, set_seed
from peft import PeftModel, prepare_model_for_kbit_training
from peft.tuners.lora.layer import LoraLayer

PERSIST_TO_DRIVE = False
RESUME_DIR = ''  # например /content/drive/MyDrive/lm_hw_expanded_runs/adapter_expanded_v1_...
if 'model' in globals() or 'RUN_DIR' in globals():
    raise RuntimeError('Инициализация уже выполнена. Для продолжения повторите ячейку 7; для нового запуска создайте новую сессию.')
if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Выберите в Colab T4 GPU.')
assert torch.__version__ == TORCH_PIN and torch.version.cuda == '13.0'
assert peft.__version__ == PINS['peft'] and np.__version__ == PINS['numpy']
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
COMMIT = 'f2768ea8029bc89bd1742e603440d6c5ac8756f4'
SOURCE_HASHES = {'checks/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'checks/autocheck.py': 'a71d2c6d589497658b61b45f72b8e7293e772beede49a0e13a5564fc4688b635', 'checks/evaluation.py': '9f92e3b3588d9872372dec99b24a29a88e50a50f481df6536974010c53c22556', 'checks/screening_v3.py': 'a94a53edb8632059de32da38c87f12054108c378bcd0a93e57f2fa382f57141f', 'data/sft_v3/selection.json': '1a4e9e6b91452d75ce54771ebe96003afd92c13839de9eb5df44f32d1874e5e4', 'data/sft_v3/pilot_config.json': 'b7fc10b036918bb0b92b8ace8f4a014e6eb90b04672cd41e9efcad575cfc7383', 'data/sft_v3/adapter_strength_selection.json': '73504f7f4af963b2919fd04aa5f3d40c50f7284f55c4a174e406fc707b9671e7', 'data/development_v3/questions.jsonl': '440228aa8ed0a2dff6c3e59b4be5155398e6c0937975fedf0b59312f5253d629', 'data/development_v3/rubric.jsonl': 'dfb1f18076e2ea7b28ce0926225fe34efb5e63b9e8ae0956aa0e93f9b8897ae7', 'data/development_v3/sources.json': '993619d62e59d60753c4f411dd3e537290b64f024c282565d78f16c8e730d88d', 'results/sft_v3_pilot/qwen3_4b_20261118T113449101907/answers.jsonl': 'e686d96d5a08692185b1687e04242abe3a4d1d1c06356ba5ff0439f4d42d01f3', 'results/sft_v3_pilot/qwen3_4b_20261118T113449101907/adapter/adapter_config.json': '2243c4784b72906e97b4c04afb0f671a51972625fe07e76ce949fa83ebf77b92', 'results/adapter_strength_v1/adapter_strength_v1_20261118T200443561241/answers.jsonl': '3bb9d2b7c9c75b56b236c161059bcc5e54d6d26da3f70f3fb34bbaaf02c30953', 'results/adapter_strength_v1/adapter_strength_v1_20261118T200443561241/run_metadata.json': 'd966fadce7ffa73d86f87830695e63082603f17c59e5923cf9cf704b82642621', 'judge/judge_prompt_v3.md': '569c99ed711c1937c4dd06a196f295e458e83b7adf71c32e4185c7758fd93ff0', 'judge/judge_response_schema_v3.json': 'e0fbf9b9a0f3e7c70ffeaa1195c687b19a3e9f81ab93f94a055155ab270aefb6', 'judge/sft_v3_pilot/judge_requests.jsonl': '3e5591918b4bf7832a10915f49477336734bf6eac6d99a5f367bd51394ef4440', 'judge/adapter_strength_v1/judge_requests.jsonl': 'cd141ad43986e7d7023417723a79c79fbb0147521f9997e381c3aa51a0e43277', 'results/sft_v3_pilot/judges/qwen_scores.jsonl': '05154ffbbb37ed7f1ec4a5d364c2f289d835bffa7e22e6d6c18a53a5003a9641', 'results/sft_v3_pilot/judges/gemma_scores.jsonl': '4ebd036c20f973241e254952b714d3b7b83506859e916755a88dcbc94d832063', 'results/adapter_strength_v1/judges/qwen_all_scales.jsonl': '6305ff87470f0e8a599d4fcaca68c2eb59a717f8e0a1d13dde331207b5c2f85d', 'results/adapter_strength_v1/judges/gemma_all_scales.jsonl': '55c1e6855e769892de8499064614c593a85f78cb8a4548d143f49f57fee42f5e'}
EMBEDDED_HELPER_SHA = 'b571d9d2ce7d713271defb1fe38917c8f08aff6d63250cedb4a83de4298b6b33'
SOURCE_ROOT = Path('/content/lm_hw_expanded_source_' + COMMIT[:7])
for relative, expected in SOURCE_HASHES.items():
    destination = SOURCE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or hashlib.sha256(destination.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/maria-igntv/lm-hw/' + COMMIT + '/' + relative
        with urllib.request.urlopen(url, timeout=60) as response:
            data = response.read()
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError('SHA mismatch: ' + relative)
        destination.write_bytes(data)
# A fresh Colab session must import exactly the pinned checks package.
if 'checks' in sys.modules:
    raise RuntimeError('Модуль checks уже импортирован из другого места. Создайте новую сессию Python.')
sys.path.insert(0, str(SOURCE_ROOT))
selection = json.loads((SOURCE_ROOT / 'data/sft_v3/selection.json').read_text())
settings = json.loads((SOURCE_ROOT / 'data/sft_v3/pilot_config.json').read_text())
candidate_selection = json.loads((SOURCE_ROOT / 'data/sft_v3/adapter_strength_selection.json').read_text())
questions = [json.loads(s) for s in (SOURCE_ROOT / 'data/development_v3/questions.jsonl').read_text().splitlines() if s.strip()]
diagnostic_metadata = json.loads((SOURCE_ROOT / 'results/adapter_strength_v1/adapter_strength_v1_20261118T200443561241/run_metadata.json').read_text())
assert diagnostic_metadata['selection'] == selection and diagnostic_metadata['generation'] == settings['generation']
assert candidate_selection['adapter_multiplier'] == 0.25
assert candidate_selection['model_id'] == selection['model_id'] == 'Qwen/Qwen3-4B-Instruct-2507'
assert candidate_selection['revision'] == selection['revision'] == 'cdbee75f17c01a7cc42f958dc650907174af0554'
GENERATION = dict(settings['generation'])
assert GENERATION == {'max_new_tokens':768, 'do_sample':True, 'temperature':0.7, 'top_p':0.8, 'top_k':20,
                      'min_p':0.0, 'repetition_penalty':1.0, 'num_beams':1, 'use_cache':True}
CONTROL_IDS = ['dev3_01', 'dev3_15', 'dev3_25']
EXPECTED_ADAPTER_SHA = candidate_selection['adapter_sha256']
assert EXPECTED_ADAPTER_SHA == '461961f38e7552aa6d4c6990b5c1aaffea5801d24b1a4d4a4b86fda61feb98f8'
question_index = {q['id']:q for q in questions}
assert len(question_index) == len(questions) == 32
SEEDS = {q['id']:settings['generation_seed'] + i for i,q in enumerate(questions, 1)}
if PERSIST_TO_DRIVE or RESUME_DIR.startswith('/content/drive/'):
    from google.colab import drive
    drive.mount('/content/drive')
OUTPUT_PARENT = Path('/content/drive/MyDrive/lm_hw_expanded_runs' if PERSIST_TO_DRIVE else '/content/lm_hw_expanded_runs')
fixed_metadata = {'source_commit':COMMIT, 'source_hashes':SOURCE_HASHES, 'embedded_helper_sha256':EMBEDDED_HELPER_SHA,
    'selection':selection, 'candidate_selection':candidate_selection, 'generation':GENERATION,
    'adapter_multiplier':0.25, 'control_ids':CONTROL_IDS, 'seeds':SEEDS,
    'versions':{p:installed(p) for p in ['torch', *PINS]},
    'scope':'Expanded development of fixed scale 0.25; not a fresh final test; no training.'}
if RESUME_DIR:
    RUN_DIR = Path(RESUME_DIR)
    metadata = json.loads((RUN_DIR / 'run_metadata.json').read_text())
    if any(metadata.get(k) != v for k,v in fixed_metadata.items()):
        raise ValueError('Возобновление остановлено: источники, версии или параметры прежнего запуска отличаются.')
    RUN_ID = metadata['run_id']
    assert RUN_DIR.name == RUN_ID
    metadata.setdefault('resumed_at_utc', []).append(datetime.now(timezone.utc).isoformat())
else:
    RUN_ID = 'adapter_expanded_v1_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
    RUN_DIR = OUTPUT_PARENT / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    metadata = {**fixed_metadata, 'run_id':RUN_ID, 'status':'prepared_not_executed',
        'started_at_utc':datetime.now(timezone.utc).isoformat(),
        'hardware':{'gpu':torch.cuda.get_device_name(0), 'python':platform.python_version(), 'cuda':torch.version.cuda}}
for relative in SOURCE_HASHES:
    target = RUN_DIR / 'source' / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SOURCE_ROOT / relative, target)
print('Источники проверены:', len(SOURCE_HASHES))
print('Каталог результата:', RUN_DIR)
print('План: 24 новых ответа + 6 контрольных; 40 ответов переиспользуются.')


## 3. Проверенные функции сохранения и подготовки пакета судье

Ячейку выполняют один раз. Она переносит прежние ответы без редактирования;
уже сохранённые новые ответы при продолжении не генерируются повторно.


In [ ]:
"""Checkpoint and blinded export for the fixed 0.25 LoRA development check.

This module is embedded verbatim into notebook 06, so Colab only downloads
already published, hash-pinned inputs. No model/API calls happen here.
"""

import hashlib
import json
import re
import zipfile
from pathlib import Path

from checks.screening_v3 import (
    export_artifacts, read_jsonl, sha256_file, summarize, validate_records,
    write_json, write_jsonl,
)

EXPANDED_PHASE = 'scale_0.25'
REUSED_IDS = ['dev3_01', 'dev3_07', 'dev3_15', 'dev3_21',
              'dev3_24', 'dev3_25', 'dev3_26', 'dev3_29']
PILOT_RUN = 'qwen3_4b_20261118T113449101907'
DIAGNOSTIC_RUN = 'adapter_strength_v1_20261118T200443561241'


def expanded_require(condition, message):
    if not condition:
        raise ValueError(message)


def expanded_signature(settings, selection, run_id):
    payload = {'settings': settings, 'selection': selection, 'run_id': run_id,
               'adapter_multiplier': 0.25, 'protocol': 'adapter_expanded_v1'}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def build_expanded_history(questions, settings, selection, pilot_rows, diagnostic_rows, run_id):
    """Retain exact historical text and tokens, with explicit origin per row."""
    qids = {q['id'] for q in questions}
    expanded_require(len(qids) == len(questions) == 32, 'Expected 32 development questions')
    baseline = [r for r in pilot_rows if r['phase'] == 'baseline']
    middle = [r for r in diagnostic_rows if r['adapter_multiplier'] == 0.25]
    expanded_require(len(baseline) == 32 and {r['id'] for r in baseline} == qids,
                     'Incomplete historical baseline')
    expanded_require(len(middle) == 8 and {r['question_id'] for r in middle} == set(REUSED_IDS),
                     'Incomplete historical scale 0.25')
    result = []
    signature = expanded_signature(settings, selection, run_id)
    for phase, source_run, rows in [('baseline', PILOT_RUN, baseline),
                                     (EXPANDED_PHASE, DIAGNOSTIC_RUN, middle)]:
        for row in rows:
            qid = row.get('question_id', row.get('id'))
            result.append({**row, 'id': qid, 'phase': phase, 'prompt_id': phase,
                           'run_id': run_id, 'source_run_id': source_run, 'origin': 'reused',
                           'model_revision': selection['revision'],
                           'adapter_multiplier': 0.0 if phase == 'baseline' else 0.25,
                           'eos_token_ids': [151645, 151643],
                           'experiment_signature': signature})
    validate_expanded_records(questions, settings, selection, run_id, result, result)
    return result


def validate_expanded_records(questions, settings, selection, run_id, rows, history, *, complete=False):
    phases = {'baseline': selection['system_prompt'], EXPANDED_PHASE: selection['system_prompt']}
    indexed = validate_records(questions, phases, rows, complete=complete)
    expected_history = {(r['prompt_id'], r['id']): r for r in history}
    expanded_require(len(expected_history) == len(history) == 40, 'Expected 40 historical answers')
    signature = expanded_signature(settings, selection, run_id)
    seeds = {q['id']: settings['generation_seed'] + i for i, q in enumerate(questions, 1)}
    for key, expected in expected_history.items():
        expanded_require(indexed.get(key) == expected, 'Historical answer was changed or removed: ' + str(key))
    for key, row in indexed.items():
        expanded_require(row.get('experiment_signature') == signature and row.get('run_id') == run_id,
                         'Saved answer belongs to a different experiment')
        expanded_require(row.get('seed') == seeds[row['id']]
                         and row['max_new_tokens'] == settings['generation']['max_new_tokens'],
                         'Saved seed or generation limit differs')
        expanded_require(row.get('model_revision') == selection['revision']
                         and row.get('phase') == row['prompt_id'], 'Saved model or phase differs')
        expected_scale = 0.0 if row['prompt_id'] == 'baseline' else 0.25
        expanded_require(row.get('adapter_multiplier') == expected_scale, 'Saved scale differs')
        if key not in expected_history:
            expanded_require(row['prompt_id'] == EXPANDED_PHASE and row.get('origin') == 'new'
                             and row.get('source_run_id') == run_id, 'Unexpected new answer provenance')
    return indexed


def generate_expanded_remaining(path, questions, settings, selection, run_id, history, generate_one, on_saved):
    """Resume without regenerating or choosing among already saved answers."""
    path = Path(path)
    rows = read_jsonl(path) if path.exists() else list(history)
    indexed = validate_expanded_records(questions, settings, selection, run_id, rows, history)
    write_jsonl(path, rows)
    signature = expanded_signature(settings, selection, run_id)
    for number, question in enumerate(questions, 1):
        key = (EXPANDED_PHASE, question['id'])
        if key in indexed:
            continue
        seed = settings['generation_seed'] + number
        answer = generate_one(question, selection['system_prompt'], seed)
        row = {**question, **answer, 'seed': seed, 'phase': EXPANDED_PHASE,
               'prompt_id': EXPANDED_PHASE, 'adapter_multiplier': 0.25,
               'run_id': run_id, 'source_run_id': run_id, 'origin': 'new',
               'model_revision': selection['revision'], 'experiment_signature': signature}
        candidate = rows + [row]
        validate_expanded_records(questions, settings, selection, run_id, candidate, history)
        write_jsonl(path, candidate)
        rows = candidate
        indexed[key] = row
        on_saved(row, sum(r['origin'] == 'new' for r in rows))
    validate_expanded_records(questions, settings, selection, run_id, rows, history, complete=True)
    return rows


def expanded_repeated_span(text):
    words = re.findall(r'\w+', text.casefold())
    for width in range(3, min(128, len(words) // 3) + 1):
        for start in range(len(words) - width * 3 + 1):
            block = words[start:start + width]
            if words[start + width:start + 3 * width] == block * 2:
                return {'period_words': width, 'start_word': start}
    return None


def export_expanded_reuse(directory, source_root, history, rubrics, sources):
    """Link 40 unchanged answers per judge to the exact original judge payload."""
    root = Path(source_root)
    rubric_index = {r['id']: r for r in rubrics}
    result = []
    for judge, model_name in [('qwen', 'Qwen3.8-27B'), ('gemma', 'gemma-4-31B-it')]:
        old = read_jsonl(root / f'results/sft_v3_pilot/judges/{judge}_scores.jsonl')
        middle = read_jsonl(root / f'results/adapter_strength_v1/judges/{judge}_all_scales.jsonl')
        prior_requests = {r['request_id']: r for r in read_jsonl(root / 'judge/sft_v3_pilot/judge_requests.jsonl')}
        diagnostic_requests = {r['request_id']: r for r in read_jsonl(root / 'judge/adapter_strength_v1/judge_requests.jsonl')}
        base = {r['question_id']: r for r in old if r['phase'] == 'baseline'}
        scaled = {r['question_id']: r for r in middle if r['adapter_multiplier'] == 0.25}
        expanded_require(len(base) == 32 and len(scaled) == 8, 'Prior score coverage differs')
        for answer in history:
            is_base = answer['prompt_id'] == 'baseline'
            score = (base if is_base else scaled)[answer['id']]
            request = (prior_requests if is_base else diagnostic_requests)[score['request_id']]
            rubric = rubric_index[answer['id']]
            basis_ids = sorted({key for c in rubric['required_criteria'] for key in c['source_ids']})
            basis = {key: {f: sources[key][f] for f in ('title', 'url', 'supports', 'limits')} for key in basis_ids}
            expected = {'question': answer['question'], 'assistant_answer': answer['answer'],
                        'case_criteria': rubric, 'source_notes': basis}
            expanded_require(json.loads(request['user_prompt']) == expected, 'Prior judged payload differs')
            expanded_require(score['judge_model'] == model_name and json.loads(score['raw_response']) == score['score'],
                             'Prior judge model or raw score differs')
            result.append({'judge': judge, 'judge_model': model_name,
                           'question_id': answer['id'], 'prompt_id': answer['prompt_id'],
                           'prior_request_id': score['request_id'], 'source_candidate_run': answer['source_run_id'],
                           'source_judge_run': score.get('source_run_id', score.get('service_batch')),
                           'score': score['score'], 'raw_response': score['raw_response'],
                           'semantic_state': score['semantic_state'],
                           'source_scores_file': f'results/sft_v3_pilot/judges/{judge}_scores.jsonl' if is_base
                           else f'results/adapter_strength_v1/judges/{judge}_all_scales.jsonl',
                           'judged_payload_verified_identical': True})
    expanded_require(len(result) == 80, 'Expected 40 reused scores per judge')
    write_jsonl(Path(directory) / 'reused_judge_scores.jsonl', result)
    return result


def export_expanded_artifacts(directory, questions, rows, history, settings, selection, run_id,
                              rubrics, sources, judge_system, judge_schema):
    directory = Path(directory)
    validate_expanded_records(questions, settings, selection, run_id, rows, history, complete=True)
    new_rows = [r for r in rows if r['origin'] == 'new']
    new_ids = {r['id'] for r in new_rows}
    expanded_require(len(new_rows) == len(new_ids) == 24, 'Expected 24 new answers')
    new_questions = [q for q in questions if q['id'] in new_ids]
    new_rubrics = [r for r in rubrics if r['id'] in new_ids]
    judge_dir = directory / 'judge'
    judge_dir.mkdir(exist_ok=True)
    count = export_artifacts(judge_dir, new_questions, {EXPANDED_PHASE: selection['system_prompt']},
                             new_rows, new_rubrics, sources, judge_system, judge_schema,
                             run_id, settings['judge_shuffle_seed'])
    config = {'status': 'prepared_not_scored', 'protocol_id': 'judge_v3.0',
              'model': 'Qwen3.8-27B', 'optional_second_model': 'gemma-4-31B-it',
              'temperature': 0, 'max_tokens': 2048, 'expected_requests': count,
              'system_prompt_path': 'judge_system_prompt.md',
              'response_schema_path': 'judge_response_schema.json',
              'response_format': {'type': 'json_schema', 'json_schema': {
                  'name': 'household_judge_v3', 'strict': True, 'schema': judge_schema}},
              'retry_policy': 'Only retry invalid/schema-inconsistent responses; preserve all attempts, errors and actual run_id. Never retry a valid low score.',
              'required_return': '24 scores with request_id, score, raw_response, judge_model, run_id; all raw attempts, error journal and model/parameter/time metadata.'}
    write_json(judge_dir / 'judge_run_config.json', config)
    with zipfile.ZipFile(judge_dir / 'for_judge.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
        for name in ['judge_requests.jsonl', 'judge_system_prompt.md',
                     'judge_response_schema.json', 'judge_run_config.json']:
            archive.write(judge_dir / name, arcname=name)
    summary = summarize(questions, {'baseline': selection['system_prompt'], EXPANDED_PHASE: selection['system_prompt']}, rows)
    for phase, group in summary['by_prompt'].items():
        phase_rows = [r for r in rows if r['prompt_id'] == phase]
        group['repeat_3_to_128_words_ids'] = [r['id'] for r in phase_rows if expanded_repeated_span(r['answer'])]
        group['repeat_after_list_counter_removal_ids'] = [r['id'] for r in phase_rows
            if expanded_repeated_span(re.sub(r'(?m)^\s*\d+[.)]\s+', '', r['answer']))]
    summary.update({'new_answers': 24, 'reused_answers': 40,
                    'quality_status': 'awaiting_external_judge_and_content_review',
                    'scope': 'Expanded development; not an independent final test.'})
    write_json(directory / 'automatic_summary.json', summary)
    indexed = {(r['prompt_id'], r['id']): r for r in rows}
    rubric_index = {r['id']: r for r in rubrics}
    report = ['# Полный development: исходная модель и LoRA 0,25\n\n',
              '32 пары, 24 новых ответа, 40 переиспользованных. Это development, не финальный тест.\n\n']
    for question in questions:
        qid = question['id']
        report.append(f"## {qid} · {question['category']}\n\n{question['question']}\n\n")
        report.extend(f"- {c['id']}: {c['text']}\n" for c in rubric_index[qid]['required_criteria'])
        for phase in ['baseline', EXPANDED_PHASE]:
            row = indexed[phase, qid]
            report.append(f"\n### {phase}\n\n")
            report.extend('> ' + line + '\n' for line in row['answer'].splitlines())
            report.append(f"\nТокенов: {row['generated_tokens']}; {row['finish_reason']}; origin={row['origin']}.\n\n")
    (directory / 'comparison.md').write_text(''.join(report), encoding='utf-8')
    return count

write_json(RUN_DIR / "run_metadata.json", metadata)
print("Функции сохранения готовы.")


## 4. Загрузите исходный ZIP SFT-пилота с весами

По умолчанию появится выбор файла. Выберите
**sft_v3_pilot_qwen3_4b_20261118T113449101907.zip** из Downloads на компьютере.
Для уже загруженного файла укажите путь в `PILOT_ZIP_PATH`.
Для пути в Drive сначала включите подключение Drive в ячейке 2.


In [ ]:
PILOT_ZIP_PATH = ''  # пусто: выбрать ZIP с компьютера; иначе путь к ZIP в /content или Drive
ADAPTER_DIR = Path('/content/lm_hw_expanded_inputs') / RUN_ID / 'adapter'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
if not (ADAPTER_DIR / 'adapter_model.safetensors').is_file():
    if PILOT_ZIP_PATH:
        user_zip = Path(PILOT_ZIP_PATH)
        if not user_zip.is_file(): raise FileNotFoundError(str(user_zip))
    else:
        from google.colab import files
        uploaded = files.upload()
        names = [n for n in uploaded if n.lower().endswith('.zip')]
        if len(names) != 1: raise ValueError('Выберите ровно один ZIP исходного SFT-пилота.')
        user_zip = Path('/content') / Path(names[0]).name
        if not user_zip.exists(): user_zip.write_bytes(uploaded[names[0]])
        del uploaded
    prefix = 'qwen3_4b_20261118T113449101907/adapter/'
    with zipfile.ZipFile(user_zip) as archive:
        for name in ['adapter_config.json', 'adapter_model.safetensors']:
            info = archive.getinfo(prefix + name)
            if info.file_size > 1_000_000_000: raise ValueError('Неожиданный размер адаптера')
            with archive.open(info) as source, (ADAPTER_DIR / name).open('wb') as dest:
                shutil.copyfileobj(source, dest, 8 << 20)
    metadata['input_pilot_zip_sha256'] = sha256_file(user_zip)
if sha256_file(ADAPTER_DIR / 'adapter_model.safetensors') != EXPECTED_ADAPTER_SHA:
    raise ValueError('SHA-256 адаптера отличается. Используйте исходный архив пилота, не tiny/smoke.')
adapter_config = json.loads((ADAPTER_DIR / 'adapter_config.json').read_text())
expected_config = json.loads((SOURCE_ROOT / 'results/sft_v3_pilot/qwen3_4b_20261118T113449101907/adapter/adapter_config.json').read_text())
assert adapter_config == expected_config
assert adapter_config['r'] == 16 and adapter_config['lora_alpha'] == 32 and adapter_config['lora_dropout'] == 0
assert not adapter_config['use_rslora'] and not adapter_config['use_dora']
metadata['adapter_sha256'] = EXPECTED_ADAPTER_SHA
write_json(RUN_DIR / 'adapter_config.json', adapter_config)
write_json(RUN_DIR / 'inference_config.json', {'model_id':selection['model_id'], 'revision':selection['revision'],
    'adapter_sha256':EXPECTED_ADAPTER_SHA, 'adapter_multiplier':0.25,
    'scaling_method':'LoraLayer.set_scale(default, 0.25), absolute relative to alpha/r=2',
    'system_prompt':selection['system_prompt'], 'generation':GENERATION,
    'status':'development_candidate_not_final'})
write_json(RUN_DIR / 'run_metadata.json', metadata)
print('SHA-256 адаптера подтверждён:', EXPECTED_ADAPTER_SHA)


## 5. Загрузка модели и одинаковых входов

Модель, NF4/FP16, eager attention и подготовка параметров совпадают с прежним
пилотом. Здесь нет оптимизатора и обновлений весов. Проверяются все 32 входа и seed.


In [ ]:
set_seed(settings['seed'])
tokenizer = AutoTokenizer.from_pretrained(selection['model_id'], revision=selection['revision'], trust_remote_code=False)
tokenizer.padding_side = 'right'
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
quant = dict(settings['quantization']); quant['bnb_4bit_compute_dtype'] = torch.float16
base_model = AutoModelForCausalLM.from_pretrained(selection['model_id'], revision=selection['revision'],
    trust_remote_code=False, dtype=torch.float16, device_map={'':0},
    quantization_config=BitsAndBytesConfig(**quant), attn_implementation='eager').eval()
assert base_model.__class__.__name__ == 'Qwen3ForCausalLM' and getattr(base_model, 'is_loaded_in_4bit', False)
eos_ids = base_model.generation_config.eos_token_id
if isinstance(eos_ids, int): eos_ids = [eos_ids]
assert set(eos_ids) == {151645, 151643}
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False})
base_model.config.use_cache = False
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), adapter_name='default',
    is_trainable=False, autocast_adapter_dtype=True).eval()
del base_model
assert set(model.peft_config) == {'default'} and not any(p.requires_grad for p in model.parameters())
assert all(p.dtype == torch.float32 for n,p in model.named_parameters() if 'lora_' in n)
assert all(p.device.type == 'cuda' for p in model.parameters())
generation_config = GenerationConfig(**GENERATION, eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id)
pilot_rows = read_jsonl(SOURCE_ROOT / 'results/sft_v3_pilot/qwen3_4b_20261118T113449101907/answers.jsonl')
diagnostic_rows = read_jsonl(SOURCE_ROOT / 'results/adapter_strength_v1/adapter_strength_v1_20261118T200443561241/answers.jsonl')
history = build_expanded_history(questions, settings, selection, pilot_rows, diagnostic_rows, RUN_ID)
historical = {(r['prompt_id'],r['id']):r for r in history}
canonical = {}
for q in questions:
    qid = q['id']
    encoded = tokenizer.apply_chat_template([{'role':'system','content':selection['system_prompt']},
        {'role':'user','content':q['question']}], add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors='pt')
    canonical[qid] = {k:v.cpu().clone() for k,v in encoded.items()}
    assert encoded['input_ids'][0].tolist() == historical['baseline',qid]['input_token_ids'], qid
    assert SEEDS[qid] == historical['baseline',qid]['seed'], qid
    assert encoded['input_ids'].shape[1] + 768 <= model.config.max_position_embeddings
for row in history:
    assert row['input_token_ids'] == canonical[row['id']]['input_ids'][0].tolist()
    assert tokenizer.decode(row['generated_token_ids'], skip_special_tokens=True) == row['answer']
    assert tokenizer.decode(row['generated_token_ids'], skip_special_tokens=False) == row['raw_answer']
write_json(RUN_DIR / 'input_audit.json', {qid:{'seed':SEEDS[qid],
    'input_token_ids':batch['input_ids'][0].tolist()} for qid,batch in canonical.items()})
layers = {n:m for n,m in model.named_modules() if isinstance(m,LoraLayer) and 'default' in m.scaling}
assert len(layers) == 252
original_scales = {n:float(m.scaling['default']) for n,m in layers.items()}
for name, layer in layers.items():
    assert not layer.merged and layer.r['default'] == 16 and layer.lora_alpha['default'] == 32
    assert original_scales[name] == 2.0
    assert tuple(inspect.signature(layer.set_scale).parameters) == ('adapter','scale')
def apply_strength(multiplier):
    if multiplier not in [0.0, 0.25]: raise ValueError('В этом запуске разрешены только 0 и 0.25')
    for name, layer in layers.items():
        layer.set_scale('default', float(multiplier))
        assert math.isclose(float(layer.scaling['default']), original_scales[name] * multiplier, rel_tol=1e-12, abs_tol=1e-12)
@contextlib.contextmanager
def isolated_rng(seed):
    py, np_state = random.getstate(), np.random.get_state()
    cpu, cuda = torch.random.get_rng_state(), torch.cuda.get_rng_state_all()
    try:
        set_seed(seed)
        yield
    finally:
        random.setstate(py); np.random.set_state(np_state)
        torch.random.set_rng_state(cpu); torch.cuda.set_rng_state_all(cuda)
@torch.inference_mode()
def prompt_logits(qid):
    batch = {k:v.clone().to('cuda') for k,v in canonical[qid].items()}
    logits = model(**batch, use_cache=False, logits_to_keep=1).logits[0,-1].float().cpu()
    if not torch.isfinite(logits).all(): raise FloatingPointError('NaN/Inf logits')
    return logits
def lora_fingerprint():
    return {n:hashlib.sha256(p.detach().cpu().contiguous().numpy().tobytes()).hexdigest()
            for n,p in model.named_parameters() if 'lora_' in n}
@torch.inference_mode()
def generate_current(qid, strength):
    apply_strength(strength)
    previous_cache = model.config.use_cache
    try:
        model.eval(); model.config.use_cache = True
        with isolated_rng(SEEDS[qid]):
            batch = {k:v.clone().to('cuda') for k,v in canonical[qid].items()}
            prefix = batch['input_ids'].shape[1]
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); started = time.perf_counter()
            output = model.generate(**batch, generation_config=generation_config)
            torch.cuda.synchronize(); elapsed = time.perf_counter() - started
            ids = output[0,prefix:].tolist()
        return {**question_index[qid], 'adapter_multiplier':strength, 'seed':SEEDS[qid],
            'answer':tokenizer.decode(ids, skip_special_tokens=True),
            'raw_answer':tokenizer.decode(ids, skip_special_tokens=False),
            'generated_token_ids':ids, 'generated_tokens':len(ids), 'eos_token_ids':eos_ids,
            'max_new_tokens':768, 'finish_reason':'eos' if ids and ids[-1] in eos_ids else ('length' if len(ids)>=768 else 'other'),
            'input_token_ids':batch['input_ids'][0].tolist(), 'elapsed_seconds':elapsed,
            'peak_allocated_gib':torch.cuda.max_memory_allocated()/1024**3,
            'peak_reserved_gib':torch.cuda.max_memory_reserved()/1024**3}
    finally:
        model.config.use_cache = previous_cache
apply_strength(0.25)
_warmup = prompt_logits(CONTROL_IDS[0])
lora_before = lora_fingerprint()
write_json(RUN_DIR / 'lora_hashes_before.json', lora_before)
write_json(RUN_DIR / 'scaling_audit.json', {'n_layers':len(layers), 'original_scales':original_scales,
    'target_multiplier':0.25, 'target_effective_scale':0.5,
    'set_scale_source_sha256':hashlib.sha256(inspect.getsource(next(iter(layers.values())).set_scale).encode()).hexdigest()})
print('Модель готова. Все 32 входа и 40 прежних текстов проверены. Новые ответы ещё не сгенерированы.')


## 6. Проверка воспроизведения — шесть контрольных ответов

Для трёх вопросов повторяются baseline и LoRA 0,25 с прежними seed.
Требуется **точное совпадение выходных токенов**, а не только похожий текст.
При расхождении запуск остановится: сохраните частичный ZIP ячейкой 8.
Не отключайте проверку и не повторяйте генерацию в поиске удачного ответа.


In [ ]:
CONTROL_SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
controls_path = RUN_DIR / ('controls_' + CONTROL_SESSION + '.jsonl')
verification = {'status':'controls_running', 'control_session':CONTROL_SESSION,
                'token_matches':{}, 'zero_vs_disabled':{}, 'scale_restoration':{}}
write_json(RUN_DIR / 'verification.json', verification)
try:
    apply_strength(0.25)
    quarter_logits = {q:prompt_logits(q) for q in CONTROL_IDS}
    apply_strength(0.0)
    zero_logits = {q:prompt_logits(q) for q in CONTROL_IDS}
    with model.disable_adapter():
        for qid in CONTROL_IDS:
            verification['zero_vs_disabled'][qid] = bool(torch.equal(zero_logits[qid], prompt_logits(qid)))
    apply_strength(0.25)
    for qid in CONTROL_IDS:
        verification['scale_restoration'][qid] = bool(torch.equal(quarter_logits[qid], prompt_logits(qid)))
    assert all(verification['zero_vs_disabled'].values()) and all(verification['scale_restoration'].values())
    controls = []
    for phase,strength in [('baseline',0.0), (EXPANDED_PHASE,0.25)]:
        for qid in CONTROL_IDS:
            row = generate_current(qid, strength)
            row.update(run_id=RUN_ID, control_session=CONTROL_SESSION, prompt_id=phase)
            controls.append(row); write_jsonl(controls_path, controls)
            matched = row['generated_token_ids'] == historical[phase,qid]['generated_token_ids']
            verification['token_matches'][phase + '/' + qid] = matched
            write_json(RUN_DIR / 'verification.json', verification)
            if not matched: raise RuntimeError('Исторические токены не воспроизвелись: ' + phase + '/' + qid + '. Сохраните ZIP ячейкой 8.')
            print('Контроль', len(controls), '/6:', phase, qid, 'MATCH', flush=True)
    verification['status'] = 'CONTROLS_PASS'
    metadata['status'] = 'controls_passed_not_complete'
    write_json(RUN_DIR / 'verification.json', verification)
    write_json(RUN_DIR / 'run_metadata.json', metadata)
    apply_strength(0.25)
    print('CONTROLS PASS. Можно генерировать недостающие 24 ответа.')
except BaseException as exc:
    metadata['status'] = 'controls_failed'
    verification['status'] = 'CONTROLS_FAILED'
    write_json(RUN_DIR / 'verification.json', verification)
    write_json(RUN_DIR / 'run_metadata.json', metadata)
    write_jsonl(RUN_DIR / 'errors.jsonl', read_jsonl(RUN_DIR / 'errors.jsonl') + [{
        'stage':'controls', 'type':type(exc).__name__, 'message':str(exc), 'traceback':traceback.format_exc()}])
    raise


## 7. Основной прогон — 24 новых ответа

Дождитесь **24/24**. Ответы сохраняются сразу после генерации.
Если ячейка прервана, но сессия жива, повторите только её: готовые ответы будут пропущены.
Для скачивания частичного результата можно выполнить ячейку 8 в любой момент после инициализации.


In [ ]:
def ensure_expanded_texts(rows):
    for row in rows:
        assert row['input_token_ids'] == canonical[row['id']]['input_ids'][0].tolist(), row['id']
        assert tokenizer.decode(row['generated_token_ids'], skip_special_tokens=True) == row['answer'], row['id']
        assert tokenizer.decode(row['generated_token_ids'], skip_special_tokens=False) == row['raw_answer'], row['id']
def generate_new_answer(question, system, seed):
    assert system == selection['system_prompt'] and seed == SEEDS[question['id']]
    return generate_current(question['id'], 0.25)
def on_new_saved(row, count):
    metadata.update(status='generating', new_answers_saved=count)
    write_json(RUN_DIR / 'run_metadata.json', metadata)
    print('Новых ответов:', count, '/24;', row['id'], row['finish_reason'], row['generated_tokens'], 'токенов', flush=True)
def run_expanded():
    assert verification['status'] in ['CONTROLS_PASS', 'TECHNICAL_PASS_NOT_QUALITY_CLAIM']
    ensure_expanded_texts(read_jsonl(RUN_DIR / 'answers.jsonl'))
    try:
        metadata['status'] = 'generating'; write_json(RUN_DIR / 'run_metadata.json', metadata)
        rows = generate_expanded_remaining(RUN_DIR / 'answers.jsonl', questions, settings, selection,
            RUN_ID, history, generate_new_answer, on_new_saved)
        ensure_expanded_texts(rows)
        lora_after = lora_fingerprint()
        write_json(RUN_DIR / 'lora_hashes_after.json', lora_after)
        verification['lora_parameters_unchanged'] = lora_before == lora_after
        verification['adapter_file_unchanged'] = sha256_file(ADAPTER_DIR / 'adapter_model.safetensors') == EXPECTED_ADAPTER_SHA
        assert verification['lora_parameters_unchanged'] and verification['adapter_file_unchanged']
        verification['status'] = 'TECHNICAL_PASS_NOT_QUALITY_CLAIM'
        metadata.update(status='complete_awaiting_export', completed_at_utc=datetime.now(timezone.utc).isoformat(),
                        total_answers=64, new_answers_saved=24, reused_answers=40)
        write_json(RUN_DIR / 'verification.json', verification)
        write_json(RUN_DIR / 'run_metadata.json', metadata)
        print('ГОТОВО: 64 ответа = 32 baseline + 32 LoRA 0,25. Из них 24 новых. Качество ещё оценивает судья.')
    except BaseException as exc:
        metadata['status'] = 'generation_interrupted_or_failed'
        write_json(RUN_DIR / 'run_metadata.json', metadata)
        write_jsonl(RUN_DIR / 'errors.jsonl', read_jsonl(RUN_DIR / 'errors.jsonl') + [{
            'stage':'generation', 'type':type(exc).__name__, 'message':str(exc), 'traceback':traceback.format_exc()}])
        raise
run_expanded()


## 8. Собрать и скачать результаты

При успешном прогоне скачиваются:

1. `adapter_expanded_v1_<timestamp>.zip` — все ответы, контроли, метаданные,
   источники, автоматические показатели и пакет судье. Этот ZIP верните для анализа.
2. `adapter_expanded_v1_<timestamp>_for_judge.zip` — **только 24 новых задания**.
   Его можно сразу передать прежнему Qwen3.8-27B с конфигурацией внутри архива.

Скачивание идёт через браузер на компьютер, обычно в **Downloads / Загрузки**.
Если браузер блокирует второй файл, разрешите множественные загрузки или скачайте
ZIP через панель **Файлы** в Colab по напечатанному пути.

Сохраните ноутбук: **Файл → Скачать → Скачать файл IPYNB**.
Полный результат не включает 125 MB весов: исходный ZIP пилота сохраните отдельно.
Частичный ZIP при ошибке содержит статус и ошибки; пакет судье создаётся только после полного PASS.


In [ ]:
def package_expanded_run():
    rows = read_jsonl(RUN_DIR / 'answers.jsonl')
    check = json.loads((RUN_DIR / 'verification.json').read_text()) if (RUN_DIR / 'verification.json').exists() else {}
    complete = metadata['status'] in ['complete_awaiting_export','complete_awaiting_judge'] and check.get('status') == 'TECHNICAL_PASS_NOT_QUALITY_CLAIM'
    judge_zip = None
    if complete:
        rubrics = read_jsonl(SOURCE_ROOT / 'data/development_v3/rubric.jsonl')
        sources = json.loads((SOURCE_ROOT / 'data/development_v3/sources.json').read_text())['sources']
        count = export_expanded_artifacts(RUN_DIR, questions, rows, history, settings, selection, RUN_ID,
            rubrics, sources, (SOURCE_ROOT / 'judge/judge_prompt_v3.md').read_text(),
            json.loads((SOURCE_ROOT / 'judge/judge_response_schema_v3.json').read_text()))
        reused = export_expanded_reuse(RUN_DIR, SOURCE_ROOT, history, rubrics, sources)
        assert count == 24 and len(reused) == 80
        metadata.update(status='complete_awaiting_judge', judge_requests=24, reused_scores_per_judge=40)
        write_json(RUN_DIR / 'run_metadata.json', metadata)
        judge_zip = RUN_DIR.parent / (RUN_ID + '_for_judge.zip')
        shutil.copy2(RUN_DIR / 'judge/for_judge.zip', judge_zip)
    else:
        print('Сохраняется ЧАСТИЧНЫЙ результат. Пакет судье не создаётся. Статус:', metadata['status'])
    archive_path = RUN_DIR.parent / (RUN_ID + '.zip')
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN_DIR.rglob('*')):
            if path.is_file(): archive.write(path, arcname=RUN_ID + '/' + str(path.relative_to(RUN_DIR)))
    print('Полный/частичный ZIP:', archive_path)
    if judge_zip: print('Пакет для Qwen3.8-27B:', judge_zip, '— 24 задания; повторно оценивать прежние 40 не нужно.')
    from google.colab import files
    files.download(str(archive_path))
    if judge_zip: files.download(str(judge_zip))
    return archive_path, judge_zip
RESULT_ZIP, JUDGE_ZIP = package_expanded_run()
